In [0]:
-- Histórico de Transações e de quantidade de Clientes por Dia, Mês e Ano

SELECT  DATE(DtCriacao) AS dtDia,
        COUNT(*) AS qtdeTransacoes,
        COUNT(DISTINCT idCliente) AS qtdeCliente
FROM workspace.tmw_loyalty.transacoes
GROUP BY ALL
ORDER BY dtDia


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
WITH tb_daily AS (
    SELECT DATE(DtCriacao) AS dtAtivacao,
           IdCliente,
           SUM(CASE WHEN  QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS QtdePontos,
           COUNT(*) AS qtdeTransacao
    FROM    workspace.tmw_loyalty.transacoes
    GROUP BY ALL
),

-- DAU (Daily Active Users / Usuários Ativos Diários)
tb_dau AS(
    SELECT  dtAtivacao,
            COUNT(DISTINCT idcliente) AS DAU
    FROM    tb_daily
    GROUP BY ALL
    ORDER BY dtAtivacao
),

-- MAU (Monthly Active Users / Usuários Ativos Mensais)
-- ARPU (Average Revenue Per User / Receita Média por Usuário)
tb_mau AS (
    SELECT  t1.dtativacao,
            t1.DAU,
            COUNT(DISTINCT t2.idcliente) AS MAU,
            SUM(QtdePontos) / COUNT(DISTINCT t2.idcliente) AS ARPU
    FROM    tb_dau AS t1
            LEFT JOIN tb_daily AS t2
                ON t1.dtAtivacao >= t2.dtAtivacao
                    AND t1.dtAtivacao - INTERVAL 28 DAYS < t2.dtativacao
    GROUP BY ALL
    ORDER BY t1.dtativacao
),

-- WAU (Weekly Active Users / Usuários Ativos Semanais)
tb_wau AS (    
    SELECT  t1.*,
            COUNT(DISTINCT t2.idcliente) AS WAU
    FROM    tb_mau AS t1
            LEFT JOIN tb_daily AS t2
                ON t1.dtAtivacao >= t2.dtAtivacao
                    AND t1.dtAtivacao - INTERVAL 7 DAYS < t2.dtativacao
    GROUP BY ALL
    ORDER BY t1.dtativacao
),

tb_mau_users AS (
    SELECT  DISTINCT
            t1.dtativacao,
            t2.idcliente
    FROM    tb_dau AS t1
            LEFT JOIN tb_daily AS t2
                ON t1.dtAtivacao >= t2.dtAtivacao
                    AND t1.dtAtivacao - INTERVAL 28 DAYS < t2.dtativacao
    ORDER BY t1.dtativacao, 
            t2.idcliente
),

-- Retenção: % usuários que permaneceram entre dois períodos e 28 dias
-- Churn: 1 - Retenção
tb_churn AS (
    SELECT  t1.dtAtivacao + INTERVAL 28 DAY AS dtChurn,
            COUNT(DISTINCT t1.idcliente) AS MAU,
            COUNT(DISTINCT t2.idcliente) AS MAUfuturo,
            COUNT(DISTINCT t2.idcliente) / count(DISTINCT t1.idcliente) AS txRetencao,
            1 - count(DISTINCT t2.idcliente) / count(DISTINCT t1.idcliente) AS txChurn
    FROM    tb_mau_users AS t1
            LEFT JOIN tb_mau_users AS t2
                ON t1.dtativacao = t2.dtativacao - INTERVAL 28 DAY
                    AND t1.idcliente = t2.idcliente
    GROUP BY ALL
    HAVING  COUNT(DISTINCT t2.idcliente) > 0
    ORDER BY 1
)

SELECT  t1.*,
        t2.txRetencao,
        t2.txChurn
FROM    tb_wau AS t1
        LEFT JOIN tb_churn AS t2
        ON t1.dtativacao = t2.dtChurn
ORDER BY dtativacao

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
WITH tb_daily AS (
    SELECT  DISTINCT
            DATE(dtCriacao) AS dtDia,
            IdCliente
    FROM    workspace.tmw_loyalty.transacoes
),

tb_lag (
    SELECT  *,
            LAG(dtDia) OVER (PARTITION BY idcliente ORDER BY dtDia) AS lagDt
    FROM    tb_daily
),

tb_cliente AS (
    SELECT  idcliente,
            AVG(DATE_DIFF(dtDia, lagDt)) AS diaRecorrencia
    FROM    tb_lag
    WHERE   lagDt IS NOT NULL
    GROUP BY ALL
),

tb_qtde AS (
    SELECT  diaRecorrencia,
            COUNT(DISTINCT idCliente) AS qtde
    FROM    tb_cliente
    GROUP BY ALL
    ORDER BY diaRecorrencia
),

tb_acum AS (
    SELECT  *,
            SUM(qtde) OVER (ORDER BY diaRecorrencia) AS qtdeAcum
    FROM    tb_qtde
)

SELECT  *,
        qtdeAcum/(SELECT MAX(qtdeAcum) FROM tb_acum) AS pctAcum,
        1 - qtdeAcum/(SELECT MAX(qtdeAcum) FROM tb_acum) AS pctSurvival
FROM    tb_acum

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
-- DATA DA FOTO 2026-06-01

SELECT IdCliente,
       count(distinct date(DtCriacao)) AS frequecia,
       count(*) AS frequeciaTransacoes,
       sum(case when QtdePontos > 0 then qtdePontos else 0 end) AS valor,
       min(date_diff('2026-06-01', DtCriacao)) AS recencia

FROM workspace.tmw_loyalty.transacoes

WHERE DtCriacao < '2026-06-01'
AND DtCriacao >= '2026-06-01' - interval 28 days

GROUP BY ALL

Databricks visualization. Run in Databricks to view.

In [0]:
%python

from sklearn import cluster
from sklearn import preprocessing

kmeans = cluster.KMeans(n_clusters=8)

min_max = preprocessing.MinMaxScaler()

df = _sqldf.toPandas()

df = df[df['IdCliente']!='163022e8-12b8-486f-8604-57d8fa0ed7e1']


X = df[['frequecia', 'valor']]
X = min_max.fit_transform(X)

kmeans.fit(X)

df['cluster'] = kmeans.predict(X)
sdf = spark.createDataFrame(df)
sdf.display()

Databricks visualization. Run in Databricks to view.